<a href="https://colab.research.google.com/github/andrewgodbout/MCS-3950-F26/blob/main/Notebooks/L06-FeatureDetection/L6_Feature_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L6 — Feature Detection: Follow-Along Examples
**MCS 3950 Computer Vision · UPEI · Fall 2026**

This notebook accompanies **Lecture 6**. It runs on the same three PEI aerial clips
you met in AL2, so every number here matches the slides.

| Section | Topic |
|---------|-------|
| 0 | Get the imagery |
| 1 | Flat, edge, corner — the auto-correlation surface |
| 2 | The structure tensor and the Harris response |
| 3 | Harris in practice — threshold and non-maximum suppression |
| 4 | What Harris actually fires on |
| 5 | Why Harris is not scale invariant |
| 6 | Image pyramids |
| 7 | Scale space and characteristic scale |
| 8 | Difference of Gaussians |
| ✏️ | **Try at home** |

> **Not submitted or graded.** Everything here is fair game for Test 1.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

def show(images, titles, cmap='gray', vmin=0, vmax=255, figw=15):
    """Show a row of images."""
    fig, axes = plt.subplots(1, len(images), figsize=(figw, 4))
    if len(images) == 1:
        axes = [axes]
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(t, fontsize=11)
        ax.axis('off')
    plt.tight_layout(); plt.show()

print("Setup complete.")

---
## 0 · Get the imagery

Same download cell as AL2 and L5 — the three clips come from the public course repository.

In [ ]:
# ── Get the aerial imagery ──────────────────────────────────────────
# Downloads three scanned PEI aerial photographs (~3 MB). Safe to re-run.
import io, os, urllib.request, zipfile

DATA_URL = ("https://raw.githubusercontent.com/andrewgodbout/"
            "MCS-3950-F26/main/Notebooks/data/AL2.zip")
NEEDED = ["pei_1935.png", "pei_1958.png", "pei_1968.png"]

if all(os.path.exists(f) for f in NEEDED):
    print("Aerial imagery already present — skipping download.")
else:
    with urllib.request.urlopen(DATA_URL, timeout=60) as resp:
        blob = resp.read()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(".")
    print(f"  {len(blob)/1e6:.1f} MB downloaded and extracted")

# resample all three onto a common 800 x 800 grid at 1 m/px, as in AL2
RES = {1935: 0.5, 1958: 1.1593543264609192, 1968: 0.5}
YEARS = [1935, 1958, 1968]

def to_grid(img, src_res, target=1.0):
    s = src_res / target
    wh = (int(round(img.shape[1]*s)), int(round(img.shape[0]*s)))
    interp = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, wh, interpolation=interp)

aerial = {y: to_grid(cv2.imread(f"pei_{y}.png", cv2.IMREAD_GRAYSCALE), RES[y])
          for y in YEARS}
side = min(min(a.shape) for a in aerial.values())
aerial = {y: a[:side, :side] for y, a in aerial.items()}

for y in YEARS:
    a = aerial[y]
    print(f"{y}: {a.shape[1]}x{a.shape[0]} px   mean {a.mean():5.1f}   std {a.std():4.1f}")

show([aerial[y] for y in YEARS], [str(y) for y in YEARS])

---
## 1 · Flat, edge, corner — the auto-correlation surface

The question the whole lecture rests on: *if I nudge this window, how much
does what I see change?*

$$E(u,v) \;=\; \sum_{x,y} w(x,y)\,\bigl[\,I(x+u,\,y+v) - I(x,y)\,\bigr]^2$$

A flat patch gives $E \approx 0$ everywhere. An edge gives a valley — you can slide
along the edge for free. A corner gives a bowl that rises whichever way you go.

In [ ]:
def structure_tensor(img, sigma=1.5, grad_sigma=1.0):
    """Gaussian-weighted second-moment matrix M = [[A, C], [C, B]]."""
    f  = cv2.GaussianBlur(img.astype(np.float64), (0, 0), grad_sigma)
    Ix = cv2.Sobel(f, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(f, cv2.CV_64F, 0, 1, ksize=3)
    A = cv2.GaussianBlur(Ix*Ix, (0, 0), sigma)
    B = cv2.GaussianBlur(Iy*Iy, (0, 0), sigma)
    C = cv2.GaussianBlur(Ix*Iy, (0, 0), sigma)
    return A, B, C

def eigenvalues(A, B, C):
    """Closed form for a symmetric 2x2. lam1 >= lam2 everywhere."""
    t = np.sqrt(np.maximum(((A - B)/2)**2 + C*C, 0))
    return (A + B)/2 + t, (A + B)/2 - t

img = aerial[1968].astype(np.float64)
A, B, C = structure_tensor(img)
lam1, lam2 = eigenvalues(A, B, C)

# Find one honest example of each type by searching the clip, rather than
# picking pixels by eye. H keeps us clear of the frame.
H = 40
inner = np.zeros(img.shape, bool); inner[H:-H, H:-H] = True

def best(score):
    s = np.where(inner, score, -np.inf)
    return np.unravel_index(np.argmax(s), s.shape)

flat   = best(-lam1)                                               # both tiny
edge   = best(np.where(lam1 > np.percentile(lam1, 99),
                       lam1/(lam2 + 1e-6), 0))                     # lopsided
corner = best(lam2)                                                # both large

WINDOWS = {"flat": flat, "edge": edge, "corner": corner}
for name, (r, c) in WINDOWS.items():
    print(f"{name:7} @ (row {r:3d}, col {c:3d})   "
          f"lambda1 = {lam1[r,c]:10,.1f}   lambda2 = {lam2[r,c]:9,.1f}   "
          f"ratio = {lam1[r,c]/max(lam2[r,c], 1e-9):8.1f}")

In [ ]:
HALF, SHIFT = 15, 8

def ssd_surface(img, r, c, half=HALF, shift=SHIFT):
    """E(u,v) over integer shifts, Gaussian-weighted."""
    patch = img[r-half:r+half+1, c-half:c+half+1]
    g = cv2.getGaussianKernel(2*half+1, half/2.0); w = g @ g.T
    E = np.zeros((2*shift+1, 2*shift+1))
    for i, u in enumerate(range(-shift, shift+1)):
        for j, v in enumerate(range(-shift, shift+1)):
            sh = img[r-half+v:r+half+1+v, c-half+u:c+half+1+u]
            E[j, i] = (w * (sh - patch)**2).sum()
    return patch, E

fig, axes = plt.subplots(2, 3, figsize=(14, 8.5))
for k, (name, (r, c)) in enumerate(WINDOWS.items()):
    patch, E = ssd_surface(img, r, c)
    axes[0, k].imshow(patch, cmap='gray', vmin=0, vmax=255)
    axes[0, k].set_title(f"{name}  @ ({c}, {r})", fontweight='bold')
    axes[0, k].axis('off')
    im = axes[1, k].imshow(E, cmap='magma', extent=[-SHIFT, SHIFT, SHIFT, -SHIFT])
    axes[1, k].set_title(f"E(u,v)   max = {E.max():,.0f}")
    axes[1, k].set_xlabel("shift u (px)"); axes[1, k].set_ylabel("shift v (px)")
    plt.colorbar(im, ax=axes[1, k], fraction=0.046)
plt.tight_layout(); plt.show()

print("Look at the three colour-bar maxima — they differ by two orders of magnitude.")

**Read the middle column carefully.** The edge surface is dark in a *stripe*, not at a
point: every shift along the hedgerow costs almost nothing. That stripe is the aperture
problem, drawn.

---
## 2 · The structure tensor and the Harris response

Taylor-expand $I$ for a small shift and $E$ collapses to a quadratic form:

$$E(u,v) \;\approx\; \begin{bmatrix} u & v\end{bmatrix} M \begin{bmatrix} u \\ v\end{bmatrix},
\qquad M = \sum w \begin{bmatrix} I_x^2 & I_xI_y \\ I_xI_y & I_y^2 \end{bmatrix}$$

The eigenvalues of $M$ classify the window. Harris and Stephens avoid computing them:

$$R \;=\; \det M - k\,(\operatorname{tr} M)^2 \;=\; \lambda_1\lambda_2 - k(\lambda_1+\lambda_2)^2$$

In [ ]:
K = 0.04

def harris_R(img, k=K, sigma=1.5):
    A, B, C = structure_tensor(img, sigma)
    return (A*B - C*C) - k*(A + B)**2      # det - k * trace^2

R_full = harris_R(img)
for name, (r, c) in WINDOWS.items():
    a, b = lam1[r, c], lam2[r, c]
    print(f"{name:7} lambda1 = {a:10,.0f}   lambda2 = {b:8,.0f}   "
          f"R = {a*b - K*(a+b)**2:15,.0f}")

core = R_full[H:-H, H:-H]
print(f"\n{100*(core < 0).mean():.1f}% of windows have R < 0  (edge-like)")
print(f"{100*(core > 0.01*core.max()).mean():.2f}% have R > 1% of the maximum  (corner-like)")

Note that the **flat** window gives $R \approx 0$ — small, but *not* negative. Flat and
edge are both rejected, for different reasons: flat has nothing to offer, edge has
something to offer in only one direction.

### 2.1 — The slide 9 self-check, done for you

Two 3×3 patches, central differences, $w = 1$, $k = 0.04$. Work them out by hand
first, then run this cell.

In [ ]:
P = np.array([[  0,   0,   0],
              [100, 100, 100],
              [100, 100, 100]], dtype=float)     # a horizontal edge

Q = np.array([[  0,   0, 100],
              [  0,   0, 100],
              [100, 100, 100]], dtype=float)     # a corner

def M_by_hand(patch, k=0.04):
    """Unweighted structure tensor from central differences over a small patch."""
    Iy, Ix = np.gradient(patch)                  # np.gradient uses central differences
    A, B, C = (Ix*Ix).sum(), (Iy*Iy).sum(), (Ix*Iy).sum()
    M   = np.array([[A, C], [C, B]])
    det, tr = A*B - C*C, A + B
    return M, det, tr, det - k*tr*tr

for name, patch in (("P (edge)", P), ("Q (corner)", Q)):
    M, det, tr, R = M_by_hand(patch)
    ev = np.linalg.eigvalsh(M)[::-1]
    print(f"{name}")
    print(f"   M = [[{M[0,0]:9,.0f}, {M[0,1]:9,.0f}],")
    print(f"        [{M[1,0]:9,.0f}, {M[1,1]:9,.0f}]]")
    print(f"   det = {det:14,.0f}    tr = {tr:10,.0f}")
    print(f"   eigenvalues = {ev[0]:,.0f}, {ev[1]:,.0f}")
    print(f"   R = det - 0.04*tr^2 = {R:14,.0f}\n")

print("Change P and Q above and re-run to test your own patches.")

---
## 3 · Harris in practice — threshold and non-maximum suppression

A raw response map is not a list of corners. Two steps turn it into one:
throw away weak responses, then keep only local maxima — the same non-maximum
suppression idea as Canny in L4, now in 2-D.

In [ ]:
def nms_peaks(R, thresh_frac=0.01, radius=5):
    """Local maxima of R above thresh_frac * R.max(). Dilation finds the max
    over a (2r+1) square, so R == dilate(R) marks the local winners."""
    k  = 2*radius + 1
    mx = cv2.dilate(R, np.ones((k, k), np.uint8))
    ys, xs = np.nonzero((R == mx) & (R > thresh_frac * R.max()))
    return xs, ys

sub = aerial[1968][0:400, 400:800]          # the detail used on the slide
R   = harris_R(sub.astype(np.float64))
xs, ys = nms_peaks(R)

fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
ax[0].imshow(sub, cmap='gray', vmin=0, vmax=255); ax[0].set_title("1968 clip (400x400)")
v = np.percentile(np.abs(R), 99.5)
im = ax[1].imshow(R, cmap='coolwarm', vmin=-v, vmax=v)
ax[1].set_title("Harris R\nred = corner, blue = edge"); plt.colorbar(im, ax=ax[1], fraction=0.046)
ax[2].imshow(sub, cmap='gray', vmin=0, vmax=255)
ax[2].plot(xs, ys, 'o', mfc='none', mec='#C8962C', mew=1.5, ms=8)
ax[2].set_title(f"threshold + NMS: {len(xs)} corners")
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

print(f"pixels in the detail : {R.size:,}")
for f in (0.001, 0.01, 0.05):
    n_thresh = int((R > f*R.max()).sum())
    n_nms    = len(nms_peaks(R, f, 5)[0])
    print(f"  R > {f:>5} x max : {n_thresh:7,} px  ->  {n_nms:5,} after NMS")
print(f"\nAt the 1% threshold we keep {100*len(xs)/R.size:.2f}% of the image as features.")

---
## 4 · What Harris actually fires on

Look again at where those 246 corners landed. They are not on the tidy
field-boundary corners — they are piled onto the woodlot canopy.

This is not a bug. Harris asks *"does this window look different from every shifted
copy of itself?"*, and dense irregular tree canopy answers **yes** almost everywhere.
Let us measure that rather than argue about it.

In [ ]:
# Split the detail into a wooded strip and an open-field strip, and compare
# corner density. (Chosen by looking at the image above, not by cherry-picking.)
regions = {"wooded (right third)": (slice(0, 400), slice(266, 400)),
           "open fields (left third)": (slice(0, 400), slice(0, 134))}

for name, (rs, cs) in regions.items():
    band = sub[rs, cs].astype(np.float64)
    Rb   = harris_R(band)
    n    = len(nms_peaks(Rb, 0.01, 5)[0])
    area = band.size / 1e4                      # in units of 100x100 px
    print(f"{name:26} {n:4d} corners   {n/area:6.1f} per 100x100 px   "
          f"std = {band.std():5.1f}")

print("\nCorner density tracks texture, not geometry.")
print("For alignment you want the sparse, stable, man-made corners.")
print("For measuring canopy, that dense response is itself the signal.")

---
## 5 · Why Harris is not scale invariant

The detector window is a fixed size. Blow the image up and the corner
grows past the window — what was a corner starts to look like an edge.

In [ ]:
r, c = WINDOWS["corner"]
CROP  = 80
patch = img[r-CROP:r+CROP, c-CROP:c+CROP]
ZOOMS = [0.5, 1, 2, 4, 8]

peak_R = []
fig, axes = plt.subplots(1, len(ZOOMS), figsize=(15, 3.4))
for ax, z in zip(axes, ZOOMS):
    interp = cv2.INTER_AREA if z < 1 else cv2.INTER_CUBIC
    big = cv2.resize(patch, None, fx=z, fy=z, interpolation=interp)
    Rz  = harris_R(big, sigma=1.5)              # window size held FIXED
    cy = cx = big.shape[0]//2
    w  = max(4, int(4*z))
    peak_R.append(Rz[cy-w:cy+w+1, cx-w:cx+w+1].max())
    h = min(50, cy)
    ax.imshow(big[cy-h:cy+h, cx-h:cx+h], cmap='gray', vmin=0, vmax=255)
    ax.add_patch(plt.Circle((h, h), 4.5, fill=False, ec='#C8962C', lw=2))
    ax.set_title(f"{z}x"); ax.axis('off')
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4))
plt.plot(ZOOMS, peak_R, 'o-', color='#B82828', lw=2.5, ms=9, mfc='white', mew=2)
plt.xscale('log', base=2); plt.xticks(ZOOMS, [f"{z}x" for z in ZOOMS])
plt.xlabel("image scale (detector window fixed)"); plt.ylabel("peak Harris R")
plt.title("The response peaks at one scale"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

best = int(np.argmax(peak_R))
for z, R_ in zip(ZOOMS, peak_R):
    print(f"  {z:>4}x : R = {R_:16,.0f}")
print(f"\npeak at {ZOOMS[best]}x; R falls {peak_R[best]/peak_R[-1]:,.0f}x by 8x")
print("That peak is not a nuisance — it says the corner HAS a scale. Section 7 measures it.")

---
## 6 · Image pyramids

Blur, halve, repeat. Two facts worth checking yourself: the whole pyramid
costs only $4/3$ of the original, and skipping the blur wrecks it.

In [ ]:
G = [aerial[1968]]
for _ in range(4):
    G.append(cv2.pyrDown(G[-1]))

L = [G[i].astype(np.float64) -
     cv2.pyrUp(G[i+1], dstsize=(G[i].shape[1], G[i].shape[0])).astype(np.float64)
     for i in range(len(G)-1)]

show(G, [f"G{i}  {g.shape[1]}x{g.shape[0]}" for i, g in enumerate(G)])
show([np.clip(l + 128, 0, 255) for l in L],
     [f"L{i} = G{i} - up(G{i+1})" for i in range(len(L))])

total = sum(g.size for g in G)
print(f"pyramid storage = {total:,} px = {total/G[0].size:.4f} x the original")
print(f"  the series 1 + 1/4 + 1/16 + ... converges to 4/3 = {4/3:.4f}")

# Laplacian pyramids are invertible - rebuild and measure the error
rec = G[-1].astype(np.float64)
for i in range(len(L)-1, -1, -1):
    rec = cv2.pyrUp(rec, dstsize=(L[i].shape[1], L[i].shape[0])) + L[i]
print(f"\nrebuild error = {np.abs(rec - G[0]).max():.2f} grey levels "
      f"(not exactly 0 - pyrDown/pyrUp round to uint8)")

In [ ]:
# Why you must blur BEFORE you subsample  (L3's aliasing, again)
naive  = aerial[1968][::16, ::16]      # just take every 16th pixel
proper = G[4]                          # pyrDown four times

show([naive, proper], ["every 16th pixel (no blur)", "pyrDown x4 (blur, then subsample)"],
     figw=9)

d = naive.astype(float) - proper.astype(float)
print(f"mean |difference| = {np.abs(d).mean():.2f} grey levels,  max = {np.abs(d).max():.0f}")
print("The aliased version invents structure that is not in the scene.")

---
## 7 · Scale space and characteristic scale

The slides treat $\sigma$ as a continuous third axis. The Laplacian of Gaussian
responds to blobs, but its raw response shrinks as $\sigma$ grows — so multiply by
$\sigma^2$ to put every scale on equal footing:

$$\text{normalised LoG} \;=\; \sigma^2\,\nabla^2 (G_\sigma * I)$$

The $\sigma$ that maximises it is the feature's **characteristic scale**.

In [ ]:
def norm_log(im, s):
    """Scale-normalised Laplacian of Gaussian."""
    return (s**2) * cv2.Laplacian(cv2.GaussianBlur(im, (0, 0), s), cv2.CV_64F, ksize=3)

img35  = aerial[1935].astype(np.float64)
SIGMAS = np.geomspace(1.0, 16.0, 24)
stack  = np.stack([norm_log(img35, s) for s in SIGMAS])

Hb = 60
si, yi, xi = np.unravel_index(np.argmax(stack[:, Hb:-Hb, Hb:-Hb]), stack[:, Hb:-Hb, Hb:-Hb].shape)
by, bx = yi + Hb, xi + Hb
curve  = stack[:, by, bx]
s_star = SIGMAS[si]

V = 85
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(img35[by-V:by+V, bx-V:bx+V], cmap='gray', vmin=0, vmax=255)
ax[0].add_patch(plt.Circle((V, V), np.sqrt(2)*s_star, fill=False, ec='#B82828', lw=2.5))
ax[0].set_title(f"1935 clip @ ({bx}, {by})"); ax[0].axis('off')
ax[1].plot(SIGMAS, curve, 'o-', color='#B82828', lw=2.3, ms=6, mfc='white')
ax[1].axvline(s_star, color='#C8962C', ls='--', lw=2)
ax[1].set_xscale('log', base=2); ax[1].set_xticks([1, 2, 4, 8, 16])
ax[1].set_xticklabels(['1', '2', '4', '8', '16'])
ax[1].set_xlabel("sigma (px)"); ax[1].set_ylabel("normalised LoG response")
ax[1].set_title("One clear peak"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"characteristic scale sigma* = {s_star:.2f} px")
print(f"blob radius = sqrt(2) * sigma* = {np.sqrt(2)*s_star:.2f} px")

raw = curve / SIGMAS**2                       # drop the normalisation
print(f"\nwithout the sigma^2 factor the peak moves to sigma = {SIGMAS[np.argmax(raw)]:.2f}")
print("The normalisation is not cosmetic - it changes the answer.")

---
## 8 · Difference of Gaussians

Computing $\nabla^2 G$ at every scale is expensive; subtracting two nearby
blurs approximates it, we've seen similar blurs in the pyramid.

$$G(k\sigma) * I - G(\sigma) * I \;\approx\; (k-1)\,\sigma^2 \nabla^2 G * I$$

In [ ]:
sigma, k = 4.0, 1.6
dog = (cv2.GaussianBlur(img35, (0, 0), k*sigma) -
       cv2.GaussianBlur(img35, (0, 0), sigma))
log = norm_log(img35, sigma)

# correlation between the two, on the interior
a, b = dog[Hb:-Hb, Hb:-Hb].ravel(), log[Hb:-Hb, Hb:-Hb].ravel()
print(f"correlation(DoG, normalised LoG) at sigma = {sigma}: "
      f"{np.corrcoef(a, b)[0,1]:.4f}")
print(f"DoG costs one subtraction; the LoG costs a Laplacian at every scale.")

v1 = np.percentile(np.abs(dog), 99); v2 = np.percentile(np.abs(log), 99)
fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(dog[by-150:by+150, bx-150:bx+150], cmap='coolwarm', vmin=-v1, vmax=v1)
ax[0].set_title(f"DoG  (sigma={sigma}, k={k})"); ax[0].axis('off')
ax[1].imshow(log[by-150:by+150, bx-150:bx+150], cmap='coolwarm', vmin=-v2, vmax=v2)
ax[1].set_title("normalised LoG"); ax[1].axis('off')
plt.tight_layout(); plt.show()

Build a DoG pyramid, find pixels that beat all 26 of their neighbours (8 in their own
level, 9 above, 9 below) and you have keypoints with a **position and a scale**. That is
the front half of SIFT — next lecture class we add the descriptor that lets you match them.

---
## ✏️ Try at home

Not graded. **A** and **B** are a good place to focus.

### A — Does $k$ matter?
Sweep `k` from 0.02 to 0.10 and plot the number of corners found (after threshold and
NMS) on the 1968 detail. Then look at *where* the corners go, not just how many.
Does a larger `k` make the detector fussier in the way the slides claim?

### B — Repeatability across 33 years
Run Harris on the same region of `aerial[1935]` and `aerial[1968]`. For each 1968
corner, is there a 1935 corner within 5 px? Report the fraction. This is the
repeatability number that actually matters — and it will be lower than you expect.
Why?

### C — Corner density as a canopy measure
The slides claim corner density tracks texture. Tile the 1968 clip into 50×50 blocks,
compute corner density per block, and display it as an image. Compare it to the
local standard deviation of the same blocks. How similar are they — and is Harris
buying you anything over plain `std`?

### D — Harris with the eigenvalues
Implement the Shi–Tomasi response $R = \min(\lambda_1, \lambda_2)$ instead of
$\det M - k(\operatorname{tr} M)^2$, and compare the corner sets. Time both.
Was avoiding the square root still worth it on your machine?